# B12 · Interactive web maps: Nepal you can zoom

> **Time:** about 2 hours  
> **You need:** Part A and Lessons B1 to B11, especially CRSs (B4), buffers (B8), counting points per district (B9), classification (B10) and map design (B11).  
> **By the end you can:**
> 1. explain what a tile basemap is, and make interactive maps from GeoDataFrames with `.explore()`,
> 2. build maps with several layers, tooltips, popups and a layer switch, and draw markers, lines and circles with `folium`,
> 3. save a map as a web page to share, and avoid the classic [lat, lon] mistake.

B11 made a map for paper. This lesson makes maps for **screens**: maps that readers can zoom, move, and question by pointing at a district. They are the quickest way to explore your own data, and a friendly way to share results with people who have no GIS software. The landslide records are **made-up practice data**, as in B9 to B11.

**Internet:** the cells run without internet, but the maps themselves need it to appear (step 1 explains why). If you are offline, run the lesson anyway and look at the saved maps later.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 24 of 39 · [Course home](../README.md)  
← Previous: [B11 · Making a beautiful map: a finished landslide map of Nepal](B11_Making_a_beautiful_map.ipynb) · Next: [C1 · What a raster is: the Kathmandu Valley as a grid of numbers](../C_Raster/C01_What_a_raster_is.ipynb) →

## 1. Static and interactive maps

The maps of B1 to B11 are **static**: fixed images, ideal for print. An **interactive** map is a web page. The reader can zoom, pan, point at a feature to see its values, and switch layers on and off. It only works on a screen.

Web maps are drawn by **Leaflet**, a JavaScript program that runs in the browser. The Python library **folium** writes that web page for you, and GeoPandas' `.explore()` uses folium behind the scenes.

Behind your data sits a **tile basemap**: a background map cut into small square images called **tiles**, 256 × 256 pixels each, with a new set for every zoom level. The browser downloads only the tiles you are looking at, from a **tile server** such as OpenStreetMap's.

So a web map needs the internet twice: to download Leaflet and to download the tiles. If only the tiles fail, the background stays grey but your data still show. With no internet at all, the map area stays empty.

In [ ]:
import geopandas as gpd
import pandas as pd
import folium
from pathlib import Path

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)

districts = gpd.read_file(DATA / "nepal_districts.geojson")
provinces = gpd.read_file(DATA / "nepal_provinces.geojson")
hq = gpd.read_file(DATA / "district_headquarters.geojson")
print(districts.crs, len(districts), len(hq))

def size_kb(path):
    """Return the size of a file in kilobytes, rounded."""
    return round(path.stat().st_size / 1024)

In [ ]:
m = districts.explore()                 # one line: a whole interactive map
m.save(OUT / "b12_districts.html")      # keep it as a web page (step 7)
print(type(m), size_kb(OUT / "b12_districts.html"), "KB")
m                                       # the last line shows the map

`.explore()` returned a `folium.Map` object, and writing `m` on the last line showed it. Try it: scroll to zoom, drag to pan, and point at a district. A **tooltip** box shows all its columns. The scale bar in the lower left changes as you zoom.

The map was also saved as an HTML web page of about 758 KB in `outputs`. Double-click it in your file browser and it opens in your web browser.

**Practice note:** in this lesson, a last line that only shows a map (such as `m`) does not count toward the line counts, like `plt.show()` in Part B.

### Practice

**Q1 · one line:** Make an interactive map of the 75 district headquarters in `hq`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
hq.explore()
```

Points appear as small circles. Point at one and its tooltip shows the `name` column, such as Kathmandu or Chautara.
</details>

**Q2 · two lines:** Select the districts of Bagmati Province (`province_code` `"NP03"`, B5) and explore only those.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
bagmati = districts[districts["province_code"] == "NP03"]
bagmati.explore()
```

The map zooms to the 13 Bagmati districts by itself: `.explore()` fits the view to the data you give it.
</details>

**Q3 · three lines:** Read `practice_landslides.csv` (made-up data), make a GeoDataFrame from its `lon` and `lat` columns (B3), and explore it with `color="crimson"`. Save the map as `b12_landslides.html` in `OUT`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
slides = pd.read_csv(DATA / "practice_landslides.csv")
points = gpd.GeoDataFrame(slides, geometry=gpd.points_from_xy(slides["lon"], slides["lat"]), crs="EPSG:4326")
m_points = points.explore(color="crimson")
m_points.save(OUT / "b12_landslides.html")
m_points
```

Forty red circles in central and eastern Nepal. `color=` gives every feature the same colour; step 2 colours by a column instead.
</details>

## 2. Styling `explore()`: colours, tooltips and backgrounds

`.explore()` takes options much like `.plot()` in B10 and B11:

| Option | What it does |
| --- | --- |
| `column=` | colours the features by this column |
| `cmap=` | the colour map, such as `"YlGnBu"` (B10) |
| `tooltip=` | the columns shown when you point at a feature (`False` for none) |
| `popup=` | the columns shown when you **click** a feature (`True` for all) |
| `tiles=` | the background: `"OpenStreetMap"` (default), `"OpenTopoMap"` (a topographic map), `"Esri.WorldImagery"` (satellite images) |
| `style_kwds=` | a dictionary of line and fill settings: `"color"` (outline), `"weight"` (line width in pixels), `"fillOpacity"` (0 to 1) |
| `legend=` | `True` (the default) adds a legend for `column=` |

Some tile providers, such as CARTO, now need a free API key. GeoPandas warns you if you choose one without it.

Areas are measured in UTM 45N, as in B4. `.explore()` accepts data in any CRS and converts it for the web by itself (step 5).

In [ ]:
districts_utm = districts.to_crs(32645)
districts_utm["area_km2"] = (districts_utm.area / 1_000_000).round(1)

m = districts_utm.explore(
    column="area_km2",                     # colour by area
    cmap="YlGnBu",                         # light to dark for small to large
    tooltip=["district", "area_km2"],      # shown when you point
    popup=True,                            # all columns, shown when you click
    tiles="OpenTopoMap",                   # a topographic background
    style_kwds={"color": "white", "weight": 0.5, "fillOpacity": 0.7},
    legend=True,
    legend_kwds={"caption": "District area (km²)"},  # the legend title
)
m.save(OUT / "b12_district_areas.html")
m

Large districts are dark blue. A `fillOpacity` of 0.7 lets the mountains of the topographic background show through. Clicking a district opens a **popup** with every column; pointing shows only the two columns you chose.

The legend in the upper right is a colour bar from about 123 to 7,965 km², the smallest and largest districts. Its tick labels can crowd together; the `caption` gives it a clear title with the unit.

### Practice

**Q1 · one line:** Show the same district areas on satellite images: `tiles="Esri.WorldImagery"`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
districts_utm.explore(column="area_km2", cmap="YlGnBu", tiles="Esri.WorldImagery")
```

The colours sit on top of real imagery. Satellite backgrounds are beautiful for context, but they compete with your colours; a pale background usually reads better.
</details>

**Q2 · two lines:** Explore only the districts larger than 3,000 km² (B5), with friendlier tooltip names: add `tooltip_kwds={"aliases": ["District", "Area (km²)"]}`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
large = districts_utm[districts_utm["area_km2"] > 3000]
large.explore(column="area_km2", tooltip=["district", "area_km2"], tooltip_kwds={"aliases": ["District", "Area (km²)"]})
```

The tooltip now reads "District" and "Area (km²)" instead of the column names. Aliases are the web map's version of a clear legend title.
</details>

**Q3 · three lines:** Join `province_names.csv` to `provinces` (B6) and explore the provinces coloured by their **name**, with `cmap="Set2"`, white outlines of weight 2, and the name as tooltip.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
names = pd.read_csv(DATA / "province_names.csv")
named = provinces.merge(names, on="province_code")
named.explore(column="province", cmap="Set2", tooltip="province", style_kwds={"color": "white", "weight": 2})
```

A text column gives one colour per category and a legend with the seven names. `Set2` is a qualitative palette: different colours with no order, as in B11's province map.
</details>

## 3. Several layers and a classified choropleth

A map with several layers is built in steps. The first `.explore()` makes the map `m`. Every later call adds to it with **`m=m`**: "draw on the map `m` that already exists". It is the web-map twin of `ax=ax` in B11, where `ax` was the map box to draw on.

`name=` gives each layer a name, and `folium.LayerControl().add_to(m)` adds a small switch, in the upper right, to turn layers on and off. Add it **last**, after all layers.

`scheme=` classifies the values with mapclassify, exactly as in B10: `"naturalbreaks"`, `"quantiles"`, `"equalinterval"`, or `"userdefined"` with your own class limits in `classification_kwds={"bins": [...]}`.

The counts per district come from the B9 recipe: a spatial join, `groupby`, then a merge so that districts with no records show 0.

In [ ]:
slides = pd.read_csv(DATA / "practice_landslides.csv")          # made-up practice data
landslides = gpd.GeoDataFrame(
    slides, geometry=gpd.points_from_xy(slides["lon"], slides["lat"]), crs="EPSG:4326"
)
joined = gpd.sjoin(landslides, districts, how="inner", predicate="within")
counts = joined.groupby("district_code").size().reset_index(name="n_landslides")
district_counts = districts.merge(counts, on="district_code", how="left")
district_counts["n_landslides"] = district_counts["n_landslides"].fillna(0).astype(int)
print(district_counts["n_landslides"].sum(), (district_counts["n_landslides"] > 0).sum())

In [ ]:
m = district_counts.explore(
    column="n_landslides", scheme="userdefined", classification_kwds={"bins": [0, 2, 5, 9]},
    cmap="OrRd", tooltip=["district", "n_landslides"],
    style_kwds={"color": "white", "weight": 0.5},
    legend_kwds={"caption": "Landslide records per district (made-up data)"},
    name="Records per district",
)
hq.explore(m=m, color="black", marker_kwds={"radius": 3}, tooltip="name", name="District headquarters")
landslides.explore(m=m, color="crimson", marker_kwds={"radius": 5},
                   tooltip=["id", "district", "trigger"], name="Landslide records")
folium.LayerControl().add_to(m)

m.save(OUT / "b12_landslide_layers.html")
m

All 40 records fall in 8 districts. The classes are the ones from B11: 0, 1 to 2, 3 to 5 and 6 to 9 records. The 69 districts without records get the palest colour.

Open the layer switch in the upper right and turn the headquarters off: the map becomes calmer. Interactive maps can hold more layers than a printed map, because the reader decides what to see. `marker_kwds={"radius": 3}` sets the size of the point circles in pixels.

### Practice

**Q1 · one line:** Draw the same choropleth with `scheme="naturalbreaks"` and `k=3` instead of your own classes.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
district_counts.explore(column="n_landslides", scheme="naturalbreaks", k=3, cmap="OrRd")
```

mapclassify chooses three classes from the data. The class limits in the legend change, and with them the story (B10): always say which scheme you used.
</details>

**Q2 · two lines:** Make a map of the counts with `cmap="OrRd"`, then add the province outlines on top with `m=` and `style_kwds={"fillOpacity": 0, "color": "black", "weight": 2}`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
m2 = district_counts.explore(column="n_landslides", cmap="OrRd")
provinces.explore(m=m2, style_kwds={"fillOpacity": 0, "color": "black", "weight": 2})
m2
```

A fill opacity of 0 draws only the outlines, like `facecolor="none"` in B11. Layer order works as in B11: later layers sit on top.
</details>

**Q3 · three lines:** Map the landslide records coloured by `trigger`, with the colours `["#d62728", "#1f77b4", "#ff7f0e"]` (earthquake, rainfall, road cut), on top of grey district outlines. Name both layers and add a layer switch.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
m3 = districts.explore(color="lightgrey", style_kwds={"color": "grey", "weight": 0.5}, name="Districts")
landslides.explore(m=m3, column="trigger", cmap=["#d62728", "#1f77b4", "#ff7f0e"], marker_kwds={"radius": 6}, name="Landslides by trigger")
folium.LayerControl().add_to(m3)
m3
```

The categories are sorted alphabetically (earthquake, rainfall, road cut), and the list of colours follows that order. Use hex codes or CSS colour names here: Matplotlib names such as `"tab:red"` mean nothing to a web browser.
</details>

## 4. folium directly: markers, lines and circles

`.explore()` maps a whole GeoDataFrame. To place single things by hand, use **folium** directly:

- `folium.Map(location=[lat, lon], zoom_start=11, tiles=...)` makes an empty map centred on a point. **Zoom levels** run from 0 (the whole world) to about 18 (single houses).
- `folium.Marker(location=[lat, lon], tooltip=..., popup=...)` is a pin.
- `folium.PolyLine(list_of_points)` is a line through a list of `[lat, lon]` pairs.
- `folium.Circle(location, radius=5000)` is a circle with a radius in **metres** on the ground, like a buffer in B8.
- `folium.CircleMarker(location, radius=10)` is a circle with a radius in **pixels** on the screen: it keeps its size when you zoom.

Every object joins the map with `.add_to(m)`.

Notice the order: folium always wants **[lat, lon]**, latitude first. Step 5 and the "When it breaks" section explain why that matters.

In [ ]:
trek = pd.read_csv(DATA / "everest_trek.csv").sort_values("order")   # approximate positions

trek_map = folium.Map(location=[27.85, 86.78], zoom_start=11, tiles="OpenTopoMap")
for _, row in trek.iterrows():                                        # one marker per stop (B11)
    folium.Marker(
        location=[row["lat"], row["lon"]],                            # latitude first
        tooltip=row["place"],
        popup=f"{row['place']}: {row['elevation_m']:,} m",
    ).add_to(trek_map)

route = [[lat, lon] for lat, lon in zip(trek["lat"], trek["lon"])]   # [lat, lon] pairs
folium.PolyLine(route, color="crimson", weight=3, tooltip="Approximate trek route").add_to(trek_map)

folium.Circle(location=[27.8047, 86.7139], radius=5000, color="purple", fill=True,
              fill_opacity=0.1, tooltip="5 km around Namche Bazaar").add_to(trek_map)
folium.CircleMarker(location=[27.6866, 86.7299], radius=10, color="black",
                    tooltip="Lukla (10 pixels)").add_to(trek_map)

trek_map.save(OUT / "b12_everest_trek.html")
print(len(route), "points on the line")
trek_map

Eight pins follow the red line from Lukla north to Everest Base Camp. Click a pin for its elevation. The purple circle shows everything within 5 km of Namche Bazaar; zoom out and it shrinks with the ground. The black ring at Lukla is a `CircleMarker`: it stays 10 pixels wide at every zoom level, so it is a symbol, not a distance.

### Practice

**Q1 · one line:** Add a marker for Everest's summit (27.9881, 86.9250 in `nepal_peaks.csv`) to `trek_map`, with a red icon: `icon=folium.Icon(color="red")`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
folium.Marker([27.9881, 86.9250], tooltip="Everest summit, 8,849 m", icon=folium.Icon(color="red")).add_to(trek_map)
trek_map
```

A red pin appears northeast of Base Camp. `folium.Icon` changes the pin's colour; useful to set one special place apart.
</details>

**Q2 · two lines:** Make a new map centred on Lukla (27.6866, 86.7299) at zoom 12, and add a 10 km circle around it.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
lukla_map = folium.Map(location=[27.6866, 86.7299], zoom_start=12)
folium.Circle([27.6866, 86.7299], radius=10000, color="purple").add_to(lukla_map)
lukla_map
```

At zoom 12 the 10 km circle is larger than the window; zoom out once to see it all. A radius in metres means the same ground distance as `buffer(10000)` in UTM (B8).
</details>

**Q3 · three lines:** Map the eight peaks of `nepal_peaks.csv` with folium: one marker each, with the name as tooltip and the height as popup. Centre the map at [28.2, 85.5] with zoom 7.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
peaks = pd.read_csv(DATA / "nepal_peaks.csv")
peaks_map = folium.Map(location=[28.2, 85.5], zoom_start=7)
for _, row in peaks.iterrows():
    folium.Marker([row["lat"], row["lon"]], tooltip=row["name"], popup=f"{row['height_m']:,} m").add_to(peaks_map)
peaks_map
```

Eight pins from Annapurna I and Dhaulagiri I in the west to Kanchenjunga in the east. The loop is the same as for the trek: data in, one marker per row.
</details>

## 5. Coordinates on web maps: WGS84 and [lat, lon]

Web maps share one convention. Your data are given in **longitude and latitude on WGS84** (EPSG:4326). The tiles are drawn in **Web Mercator** (EPSG:3857), and Leaflet converts your data on the fly. Web Mercator is fine for viewing but stretches areas far from the equator (B4), so never **measure** on it.

`.explore()` knows about CRSs. Give it data in UTM 45N and it converts to EPSG:4326 for you.

**folium** knows nothing about CRSs. It wants degrees, in the order **[lat, lon]**. Shapely and GeoPandas use the opposite order, (x, y) = (lon, lat), as you learned in B1 and B2. So when you take a point from a GeoDataFrame to folium, write **`[point.y, point.x]`**.

The example draws the same 10 km zone around Kathmandu's headquarters twice: as a buffer made in UTM (B8) and as a folium circle.

In [ ]:
ktm_hq = hq[hq["name"] == "Kathmandu"]
point = ktm_hq.geometry.iloc[0]
print("shapely (x, y):", round(point.x, 4), round(point.y, 4))
print("folium [lat, lon]:", [round(point.y, 4), round(point.x, 4)])

zone_utm = ktm_hq.to_crs(32645).buffer(10000)             # 10 km buffer in metres (B8)
print(zone_utm.crs.name)

ktm_map = zone_utm.explore(color="orange", tooltip=False)   # explore converts UTM for you
folium.Circle([point.y, point.x], radius=10000, color="black", fill=False,
              tooltip="folium.Circle, 10 km").add_to(ktm_map)
folium.Marker([point.y, point.x], tooltip="Kathmandu headquarters").add_to(ktm_map)

ktm_map.save(OUT / "b12_kathmandu_10km.html")
ktm_map

The point prints as (85.3228, 27.7129) in shapely's order and as [27.7129, 85.3228] for folium. The orange buffer was made in UTM zone 45N, yet `.explore()` placed it correctly, and the black folium circle lies right on its edge. Two tools, one 10 km zone.

The buffer's edge is a polygon with many corners (B8), so the two outlines can differ by a few metres. That is a drawing detail, not a measurement.

### Practice

**Q1 · one line:** Print the geometry of `ktm_hq` in UTM 45N. Could you give these two numbers to folium?

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(ktm_hq.to_crs(32645).geometry.iloc[0])
```

`POINT (334640.4... 3066523.8...)`: metres east and north in UTM 45N. folium would read the second number, over 3 million, as a latitude, which is meaningless. folium needs degrees: convert to EPSG:4326 first, or let `.explore()` do it.
</details>

**Q2 · two lines:** Print the [lat, lon] pair of the Chautara headquarters, rounded to 4 decimal places, ready for folium.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
chautara = hq[hq["name"] == "Chautara"].geometry.iloc[0]
print([round(chautara.y, 4), round(chautara.x, 4)])
```

`[27.7865, 85.7162]`: `.y` comes first because folium wants latitude first.
</details>

**Q3 · three lines:** Put a small `CircleMarker` (radius 3) at each of the 75 headquarters on a folium map centred at [28.3, 84.1], zoom 7, with the name as tooltip. Loop with `zip(hq.geometry, hq["name"])` (A12).

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
hq_map = folium.Map(location=[28.3, 84.1], zoom_start=7)
for pt, name in zip(hq.geometry, hq["name"]):
    folium.CircleMarker([pt.y, pt.x], radius=3, tooltip=name).add_to(hq_map)
hq_map
```

75 dots across Nepal. `[pt.y, pt.x]` turns each shapely point into folium's order; `hq.explore()` does the same job in one line, but now you know what happens inside it.
</details>

## 6. Groups of markers and richer popups

With folium you can build your own layers. A **`folium.FeatureGroup(name=...)`** is a layer that holds many markers. Add markers to the group, add the group to the map, and the layer switch lists it by name.

Popups and tooltips accept a little **HTML**, the language of web pages. Two tags are enough for most popups:

- `<b>text</b>` makes text **bold**,
- `<br>` starts a new line.

Below, the made-up landslide records are split into one layer per trigger, each with its own colour, and each popup shows the record's id, district and area.

In [ ]:
trigger_colours = {"rainfall": "#1f77b4", "road cut": "#ff7f0e", "earthquake": "#d62728"}

slide_map = folium.Map(location=[28.0, 85.0], zoom_start=7, tiles="OpenTopoMap")
for trigger, colour in trigger_colours.items():
    subset = landslides[landslides["trigger"] == trigger]
    group = folium.FeatureGroup(name=f"{trigger} ({len(subset)})")
    for _, row in subset.iterrows():
        popup_html = f"<b>{row['id']}</b><br>{row['district']}<br>{row['area_m2']:,} m²"
        folium.CircleMarker([row["lat"], row["lon"]], radius=6, color=colour, fill=True,
                            fill_opacity=0.8, popup=popup_html).add_to(group)
    group.add_to(slide_map)
folium.LayerControl(collapsed=False).add_to(slide_map)

slide_map.save(OUT / "b12_landslides_by_trigger.html")
slide_map

The layer switch stays open (`collapsed=False`). It lists the background tiles, then three layers with their counts in brackets: rainfall (29), road cut (8) and earthquake (3). Turn rainfall off and the few earthquake records stand out. Click any circle: the popup shows the id in bold, then the district and the area on separate lines.

### Practice

**Q1 · one line:** Check the counts in the layer names: count the records per trigger in `landslides`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
landslides["trigger"].value_counts()
```

rainfall 29, road cut 8, earthquake 3: the same as the layer names. A layer name with a count is only useful if the count is right, so check it once.
</details>

**Q2 · two lines:** Build the popup text for the first record of `landslides`, with its id in bold and its date on a second line, and print it.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
first = landslides.iloc[0]
print(f"<b>{first['id']}</b><br>{first['date']}")
```

`<b>LS001</b><br>2023-08-18`. In a notebook it prints as plain text; in a popup the browser turns the tags into bold text and a line break.
</details>

**Q3 · three lines:** Add a fourth layer to `slide_map`: the records of 10,000 m² or more, as black `CircleMarker`s of radius 12 with no fill, named `"10,000 m² or more"`. Add a new layer switch too.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
big = folium.FeatureGroup(name="10,000 m² or more")
for _, row in landslides[landslides["area_m2"] >= 10000].iterrows():
    folium.CircleMarker([row["lat"], row["lon"]], radius=12, color="black", fill=False).add_to(big)
big.add_to(slide_map)
folium.LayerControl().add_to(slide_map)
slide_map
```

Black rings now circle the largest records, and you can switch them on and off. A map can hold only one useful layer switch, so the new one replaces the old. In your own work, add every layer first and the switch last.
</details>

## 7. Saving and sharing

`m.save(path)` writes the whole map into **one HTML file**. Anyone can open it in a web browser, with no Python and no GIS software. Share it by email, on a shared drive, or on a website (GitHub Pages hosts such files for free).

Inside the file are your data, written as GeoJSON text, and instructions for the browser. Leaflet and the tiles are **not** inside: the browser downloads them when the file opens. So the reader needs the internet, as you did.

The file grows with your data. Detailed boundaries make heavy pages that open slowly on a phone. For the web, **simplify** the shapes first: `simplify(1000)` in UTM removes corners closer than about 1,000 m to the outline. Each district is simplified on its own, so small gaps can open along shared borders; that is fine for an overview, not for analysis.

Maps also make notebooks large. Before sharing a notebook, use Edit → Clear Outputs of All Cells.

In [ ]:
for path in sorted(OUT.glob("b12_*.html")):         # every map saved so far (A7)
    print(f"{path.name:32} {size_kb(path):5} KB")

In [ ]:
light = districts_utm.copy()
light["geometry"] = light.simplify(1000)            # tolerance in metres: we are in UTM
light.explore(tooltip=["district", "area_km2"]).save(OUT / "b12_districts_light.html")

print(size_kb(OUT / "b12_districts.html"), "KB before,", size_kb(OUT / "b12_districts_light.html"), "KB after")
html_text = (OUT / "b12_districts_light.html").read_text(encoding="utf-8")
print(html_text.splitlines()[0])      # the first line of the file

The district maps are the heaviest files, because they carry 77 detailed outlines. Simplifying to 1 km shrank the file from 758 KB to 164 KB, and at national scale the map looks the same. The file starts with `<!DOCTYPE html>`: it is a plain web page.

**Static maps with a background.** Sometimes you want a basemap behind a **static** map for a report. The `contextily` library adds tiles to a Matplotlib map. It needs the internet, so it is shown here as text only. Try it at home when you are online:

```python
import contextily as cx
ax = districts.to_crs(3857).plot(facecolor="none", edgecolor="black")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
```

The data go to Web Mercator (EPSG:3857) first, because that is the CRS of the tiles. Credit the tile provider in your source note (B11).

### Practice

**Q1 · one line:** Print the size of the trek map file in KB.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(size_kb(OUT / "b12_everest_trek.html"))
```

About 13 KB: eight markers and one line are tiny next to 77 district outlines.
</details>

**Q2 · two lines:** Check that your data really are inside the saved page: read `b12_landslide_layers.html` as text and test whether `"Dolakha"` and `"LS001"` appear in it.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
layers_text = (OUT / "b12_landslide_layers.html").read_text(encoding="utf-8")
print("Dolakha" in layers_text, "LS001" in layers_text)
```

`True True`: the district names and the record ids travel inside the file. So be careful what you share: a web map hands over the data, not only a picture of it.
</details>

**Q3 · three lines:** Simplify `districts_utm` with a tolerance of 3,000 m, save it as `b12_districts_3km.html`, and print the sizes of the original, 1 km and 3 km files.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
rough = districts_utm.copy()
rough["geometry"] = rough.simplify(3000)
rough.explore().save(OUT / "b12_districts_3km.html")
print([size_kb(OUT / name) for name in ["b12_districts.html", "b12_districts_light.html", "b12_districts_3km.html"]])
```

Each step makes the file smaller, but zoom in and the 3 km outlines look clumsy and gaps appear along the borders. Choose the tolerance for the zoom levels your readers will use.
</details>

## 8. When it breaks: latitude and longitude swapped

The most common web-map mistake raises **no error at all**. Below, two markers should both sit on Kathmandu. The red one has its numbers in the wrong order: [lon, lat] instead of [lat, lon]. Run the cell and look for both markers.

In [ ]:
lat, lon = 27.7129, 85.3228                          # Kathmandu headquarters, rounded

check_map = folium.Map(location=[50, 60], zoom_start=2)          # the whole of Asia and Europe
folium.Marker([lat, lon], tooltip="right: [lat, lon]").add_to(check_map)
folium.Marker([lon, lat], tooltip="swapped: [lon, lat]", icon=folium.Icon(color="red")).add_to(check_map)
check_map.save(OUT / "b12_swapped_marker.html")
check_map

The blue marker sits in Nepal. The red one sits at the very top edge of the world map, far north in the Arctic Ocean, because folium read 85.3 as a **latitude**. Both numbers are valid latitudes, so folium cannot know you made a mistake.

**How to spot it:** a marker that is missing, or far from where it should be; a map that opens in the wrong part of the world; points in a line along the top or side of the map. With GeoPandas data, `.explore()` avoids the problem, because it reads the order from the geometry.

**How to catch it in code:** check every pair before you map it, with an `assert` (A12). Nepal's districts span about 26.3° to 30.4° N and 80.1° to 88.2° E (`districts.total_bounds`). Run the cell.

In [ ]:
west, south, east, north = districts.total_bounds

def in_nepal(lat, lon):
    """Return True if a [lat, lon] pair lies inside Nepal's bounding box."""
    return south <= lat <= north and west <= lon <= east

assert in_nepal(27.7129, 85.3228), "the right pair should pass"
assert in_nepal(85.3228, 27.7129), "this pair is not in Nepal: are lat and lon swapped?"

The last line reads `AssertionError: this pair is not in Nepal: are lat and lon swapped?`. The first assert passed silently; the second caught the swapped pair, with your own message pointing at the likely cause.

The fix is always the order: `[lat, lon]` for folium, or `[point.y, point.x]` from a shapely point. Name your variables `lat` and `lon`, never `a` and `b`, and the mistake becomes visible when you read the line.

**Try it:** swap the numbers in the second `assert` so it passes.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines, not counting a last line that only shows the map. Plan the map in words first (purpose, layers, what the tooltips show), then write the code.

### Review 1 · The trek, ready to share

Make a folium map of the Everest trek on `"OpenTopoMap"` tiles. Colour each stop's pin by elevation: red above 4,000 m, blue otherwise (a made-up threshold for practice). Draw the route as a line and a 5 km circle around Namche Bazaar. Save it as `b12_trek_review.html` and print its size in KB.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
review_map = folium.Map(location=[27.85, 86.78], zoom_start=11, tiles="OpenTopoMap")
for _, row in trek.iterrows():
    colour = "red" if row["elevation_m"] > 4000 else "blue"
    folium.Marker([row["lat"], row["lon"]], tooltip=f"{row['place']} ({row['elevation_m']:,} m)",
                  icon=folium.Icon(color=colour)).add_to(review_map)
folium.PolyLine([[lat, lon] for lat, lon in zip(trek["lat"], trek["lon"])], color="crimson").add_to(review_map)
folium.Circle([27.8047, 86.7139], radius=5000, color="purple").add_to(review_map)
review_map.save(OUT / "b12_trek_review.html")
print(size_kb(OUT / "b12_trek_review.html"), "KB")
review_map
```

Four stops are above 4,000 m (Dingboche, Lobuche, Gorak Shep and Base Camp), so four pins are red. `"red" if ... else "blue"` chooses the colour in one line; an `if`/`else` block (A4) does the same job. The file is only about 11 KB.
</details>

### Review 2 · A density explorer

Add a column `density_per_100km2` to a copy of `district_counts` in UTM 45N (records ÷ area in km² × 100, as in B10). Explore it with `scheme="naturalbreaks"`, `k=4`, friendly tooltip aliases, and the district headquarters as a second layer. Add a layer switch and save the map.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
density = district_counts.to_crs(32645)
density["area_km2"] = density.area / 1_000_000
density["density_per_100km2"] = (density["n_landslides"] / density["area_km2"] * 100).round(2)

dmap = density.explore(column="density_per_100km2", scheme="naturalbreaks", k=4, cmap="Reds",
                       tooltip=["district", "n_landslides", "density_per_100km2"],
                       tooltip_kwds={"aliases": ["District", "Records", "Records per 100 km²"]},
                       name="Records per 100 km² (made-up data)")
hq.explore(m=dmap, color="black", marker_kwds={"radius": 3}, tooltip="name", name="Headquarters")
folium.LayerControl().add_to(dmap)
dmap.save(OUT / "b12_density.html")
dmap
```

Try `scheme="quantiles"` too: with 69 of 77 districts at 0, four classes with equal numbers of districts are impossible, and mapclassify warns you. Honest classes matter on a web map as much as on paper (B10). The density was calculated in UTM, and `.explore()` converted the result for the web.
</details>

### Review 3 · Headquarters in the Kathmandu Valley box

Select the headquarters inside the approximate Kathmandu Valley box (lon 85.2 to 85.55, lat 27.55 to 27.82) with `.cx` (B5). Make a folium map with a `folium.Rectangle(bounds=[[south, west], [north, east]])` for the box and a marker for each selected headquarters. Fit the view to the box with `m.fit_bounds(...)`, which also takes [lat, lon] corners, and save it.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
valley_hq = hq.cx[85.2:85.55, 27.55:27.82]
print(list(valley_hq["name"]))

box = [[27.55, 85.2], [27.82, 85.55]]                  # [[south, west], [north, east]]
valley_map = folium.Map(location=[27.7, 85.37], zoom_start=11)
folium.Rectangle(bounds=box, color="black", fill=False, tooltip="Approximate Kathmandu Valley box").add_to(valley_map)
for pt, name in zip(valley_hq.geometry, valley_hq["name"]):
    folium.Marker([pt.y, pt.x], tooltip=name).add_to(valley_map)
valley_map.fit_bounds(box)
valley_map.save(OUT / "b12_valley_box.html")
valley_map
```

Three headquarters fall in the box: Bhaktapur, Kathmandu and Lalitpur. Note the two orders side by side: `.cx` takes x (longitude) first, while folium's `bounds` take latitude first. Writing `[[south, west], [north, east]]` in a comment helps.
</details>

### Review 4 · Check before you map

Write a function `bad_pairs(pairs)` that returns every [lat, lon] pair outside Nepal's bounding box, using `in_nepal` from section 8. Test it on the trek stops plus one swapped pair, `[86.7299, 27.6866]`. Assert that exactly one bad pair is found, then map only the good pairs as a line.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
def bad_pairs(pairs):
    """Return the [lat, lon] pairs that are not inside Nepal's bounding box."""
    return [pair for pair in pairs if not in_nepal(pair[0], pair[1])]

pairs = [[lat, lon] for lat, lon in zip(trek["lat"], trek["lon"])] + [[86.7299, 27.6866]]
bad = bad_pairs(pairs)
print(bad)
assert len(bad) == 1, "expected exactly one swapped pair"

good = [pair for pair in pairs if pair not in bad]
check_line = folium.Map(location=[27.85, 86.78], zoom_start=10)
folium.PolyLine(good, color="crimson").add_to(check_line)
check_line
```

The function finds `[[86.7299, 27.6866]]`, Lukla with its numbers swapped, and the assert confirms that the eight real stops pass. A check like this belongs in front of every map built from typed or downloaded coordinates.
</details>

## Recap

- An interactive map is a web page drawn by Leaflet over a **tile basemap**; it needs the internet for Leaflet and for the tiles.
- `gdf.explore()` makes a map in one line; `column`, `cmap`, `scheme`, `tooltip`, `popup`, `tiles`, `style_kwds` and `legend` style it, as `.plot()` did in Part B.
- Add layers with `m=m` and `name=`, then `folium.LayerControl().add_to(m)` last. `folium.Marker`, `PolyLine`, `Circle` (metres) and `CircleMarker` (pixels) place things by hand.
- `.explore()` converts any CRS for you; folium wants degrees in **[lat, lon]** order, so use `[point.y, point.x]`. A swapped pair raises no error: check with an `assert`.
- `m.save(path)` writes one HTML file to share. Your data travel inside it; simplify big layers so the page stays light.

### Words you learned

| Word | Meaning |
| --- | --- |
| static / interactive map | A fixed image / a web page you can zoom, pan and question |
| Leaflet | The JavaScript program that draws web maps in a browser |
| folium | The Python library that writes Leaflet web maps |
| `.explore()` | GeoPandas' one-line interactive map, built with folium |
| tile / tile basemap | One small square image of a background map / the background made of them |
| tile server | A website that sends map tiles, such as OpenStreetMap's |
| zoom level | How far in you are: 0 is the whole world, about 18 is single houses |
| tooltip | Text that appears when you point at a feature |
| popup | A box that opens when you click a feature |
| layer control | The switch that turns map layers on and off |
| `m=m` | Draw on the existing map `m`, like `ax=ax` in Matplotlib |
| marker | A pin or symbol at one point on a web map |
| `FeatureGroup` | A folium layer that holds many markers |
| HTML | The language of web pages; `<b>` for bold, `<br>` for a new line |
| Web Mercator | EPSG:3857, the CRS of web map tiles; fine for viewing, not for measuring |
| simplify | Remove corners from shapes so files are smaller, within a tolerance |
| `contextily` | A library that adds web tiles behind a static Matplotlib map |

## Next

**This is the end of Part B · Vector GIS.** You can now read, project, select, join, cut, count and map vector data, on paper and on screen. **Part C · Rasters begins with C1 · What a raster is:** grids of cells such as elevation models and satellite images, where every cell holds a value.

<!-- pygile-nav -->
**PyGILE Course** · lesson 24 of 39 · [Course home](../README.md)  
← Previous: [B11 · Making a beautiful map: a finished landslide map of Nepal](B11_Making_a_beautiful_map.ipynb) · Next: [C1 · What a raster is: the Kathmandu Valley as a grid of numbers](../C_Raster/C01_What_a_raster_is.ipynb) →